# Assemble Text Datasets

This notebook assembles the datasets in a text form so to be used for training LLMs.

## Setup

In [39]:
import os
import sys
from typing import Mapping, Literal, Callable, List, ClassVar, Any, Tuple, Dict
from collections import Counter, defaultdict
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, MACCSkeys, rdFMCS, Draw
from rdkit import RDLogger
from rdkit import rdBase
from sklearn.model_selection import train_test_split, GroupShuffleSplit

if 'ipykernel' in sys.modules:
    from tqdm.auto import tqdm  # for notebooks
else:
    from tqdm import tqdm

def safe_display(*args):
    """Displays content only if running in a Jupyter notebook."""
    if 'ipykernel' in sys.modules:
        from IPython.display import display
        display(*args)
    else:
        print(*args)

# Disable the RDKit warnings that pop up when RDKit fails to create molecules
RDLogger.DisableLog("rdApp.*")
blocker = rdBase.BlockLogs()

data_dir = os.path.join(os.getcwd(), 'data')

In [40]:
import sys

sys.path.append(os.path.join(os.getcwd(), 'protac_splitter'))

from protac_splitter.protac_cheminformatics import (
    reassemble_protac,
)

In [41]:
ds = {
    'standard': {
        'train': pd.read_csv(os.path.join(data_dir, 'datasets', 'standard', 'train.csv')),
        'test': pd.read_csv(os.path.join(data_dir, 'datasets', 'standard', 'test.csv'))
    },
    'hardest': {
        'train': pd.read_csv(os.path.join(data_dir, 'datasets', 'hardest', 'train.csv')),
        'test': pd.read_csv(os.path.join(data_dir, 'datasets', 'hardest', 'test.csv'))
    },
    'e3_unique': {
        'train': pd.read_csv(os.path.join(data_dir, 'datasets', 'e3_unique', 'train.csv')),
        'test': pd.read_csv(os.path.join(data_dir, 'datasets', 'e3_unique', 'test.csv'))
    },
    'linker_unique': {
        'train': pd.read_csv(os.path.join(data_dir, 'datasets', 'linker_unique', 'train.csv')),
        'test': pd.read_csv(os.path.join(data_dir, 'datasets', 'linker_unique', 'test.csv'))
    },
    'poi_unique': {
        'train': pd.read_csv(os.path.join(data_dir, 'datasets', 'poi_unique', 'train.csv')),
        'test': pd.read_csv(os.path.join(data_dir, 'datasets', 'poi_unique', 'test.csv'))
    }
}

In [42]:
relevant_cols = [c for c in ds['standard']['train'].columns if 'smiles' in c.lower()]
for i, row in ds['standard']['train'][relevant_cols].head().iterrows():
    safe_display(row.to_dict())

{'PROTAC SMILES': 'C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NC(C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'E3 Binder SMILES': 'CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'E3 Binder SMILES with direction': '[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'Linker SMILES': 'COCCOCCOCCN(C)C(=O)CCN',
 'Linker SMILES with direction': '[*:2]COCCOCCOCCN(C)C(=O)CCN[*:1]',
 'POI Ligand SMILES': 'C=CC(=O)N1CCN(c2ncnc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'POI Ligand SMILES with direction': '[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1'}

{'PROTAC SMILES': 'C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCC(=O)N[C@H](C(=O)N3C[C@H](O)C[C@H]3C(=O)N[C@@H](C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'E3 Binder SMILES': 'CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'E3 Binder SMILES with direction': '[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'Linker SMILES': 'COCCOCCOCCN(C)C(=O)CCN',
 'Linker SMILES with direction': '[*:2]COCCOCCOCCN(C)C(=O)CCN[*:1]',
 'POI Ligand SMILES': 'C=CC(=O)N1CCN(c2ncnc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'POI Ligand SMILES with direction': '[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1'}

{'PROTAC SMILES': 'C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCOCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NC(C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'E3 Binder SMILES': 'CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'E3 Binder SMILES with direction': '[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'Linker SMILES': 'OCCOCCOCCOCCN(C)C(=O)CCN',
 'Linker SMILES with direction': '[*:2]OCCOCCOCCOCCN(C)C(=O)CCN[*:1]',
 'POI Ligand SMILES': 'C=CC(=O)N1CCN(c2ncnc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'POI Ligand SMILES with direction': '[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1'}

{'PROTAC SMILES': 'C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCOCC(=O)N[C@H](C(=O)N3C[C@H](O)C[C@H]3C(=O)N[C@@H](C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'E3 Binder SMILES': 'CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'E3 Binder SMILES with direction': '[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'Linker SMILES': 'OCCOCCOCCOCCN(C)C(=O)CCN',
 'Linker SMILES with direction': '[*:2]OCCOCCOCCOCCN(C)C(=O)CCN[*:1]',
 'POI Ligand SMILES': 'C=CC(=O)N1CCN(c2ncnc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'POI Ligand SMILES with direction': '[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1'}

{'PROTAC SMILES': 'C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCOCCOCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NC(C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'E3 Binder SMILES': 'CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'E3 Binder SMILES with direction': '[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 'Linker SMILES': 'OCCOCCOCCOCCOCCN(C)C(=O)CCN',
 'Linker SMILES with direction': '[*:2]OCCOCCOCCOCCOCCN(C)C(=O)CCN[*:1]',
 'POI Ligand SMILES': 'C=CC(=O)N1CCN(c2ncnc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'POI Ligand SMILES with direction': '[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1'}

In [43]:
def remove_stereo(smiles: str) -> str:
    try:
        mol = Chem.MolFromSmiles(smiles)
        Chem.rdmolops.RemoveStereochemistry(mol)
        return Chem.MolToSmiles(mol)
    except Exception as e:
        # print(e)
        return np.nan

def randomize_smiles(smiles: str) -> str:
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Chem.MolToSmiles(mol, canonical=False, doRandom=True)
    except Exception as e:
        # print(e)
        return np.nan


def canonize_smiles(smiles: str) -> str:
    try:
        return Chem.MolToSmiles(Chem.MolFromSmiles(smiles), canonical=True)
    except:
        return np.nan


def levenshtein_distance(s1: str, s2: str) -> int:
    """ Returns the Levenshtein distance between two strings.
    
    Args:
        s1: The first string.
        s2: The second string.
    
    Returns:
        The Levenshtein distance between the two strings.
    """
    # Create a matrix of zeros with dimensions len(s1) + 1 x len(s2) + 1
    matrix = np.zeros((len(s1) + 1, len(s2) + 1))
    # Fill the first row with the index of each character in s1
    for i in range(len(s1) + 1):
        matrix[i, 0] = i
    # Fill the first column with the index of each character in s2
    for j in range(len(s2) + 1):
        matrix[0, j] = j
    # Iterate over the matrix and fill in the values
    for i in range(1, len(s1) + 1):
        for j in range(1, len(s2) + 1):
            # If the characters are the same, the cost is 0
            if s1[i - 1] == s2[j - 1]:
                cost = 0
            else:
                cost = 1
            # Fill in the matrix with the minimum of the three possible values
            matrix[i, j] = min(
                matrix[i - 1, j] + 1,
                matrix[i, j - 1] + 1,
                matrix[i - 1, j - 1] + cost
            )
    # Return the bottom right value of the matrix
    return matrix[-1, -1]


def get_ordered_substruct(
        protac_smiles: str,
        substruct1: str,
        substruct2: str,
) -> Tuple[str, str]:
    """Returns the first substructure that is found in the PROTAC SMILES string.
    
    Args:
        protac_smiles: The PROTAC SMILES string.
        substruct1: The first substructure string.
        substruct2: The second substructure string.

    Returns:
        The first substructure that is found in the PROTAC SMILES string.
    """
    # Remove stereochemistry and attachment points from the SMILES strings
    protac_smiles = remove_stereo(protac_smiles)
    substruct1_nodir = remove_stereo(substruct1).replace('[*:1]', '').replace('[*:2]', '')
    substruct2_nodir = remove_stereo(substruct2).replace('[*:1]', '').replace('[*:2]', '')

    # Remove all digits from the PROTAC SMILES string
    protac_smiles = ''.join([i for i in protac_smiles if not i.isdigit()])
    substruct1_nodir = ''.join([i for i in substruct1_nodir if not i.isdigit()])
    substruct2_nodir = ''.join([i for i in substruct2_nodir if not i.isdigit()])

    # Get a proportion of the PROTAC SMILES string that is the same length as
    # the substructure strings
    protac_sub1 = protac_smiles[:len(substruct1_nodir)]
    protac_sub2 = protac_smiles[:len(substruct2_nodir)]

    # # Check how "similar" the protac_sub1 string is to substruct1_nodir string
    # sub1_sim = sum([1 for i, j in zip(protac_sub1, substruct1_nodir) if i == j]) / len(substruct1_nodir)
    # # Check how "close" the protac_sub2 string is to substruct2_nodir string
    # sub2_sim = sum([1 for i, j in zip(protac_sub2, substruct2_nodir) if i == j]) / len(substruct2_nodir)
    # # Return the substructure that is more similar to the PROTAC SMILES string
    # return substruct1 if sub1_sim > sub2_sim else substruct2

    # Check how "similar" the protac_sub1 string is to substruct1_nodir string
    sub1_dist = levenshtein_distance(protac_sub1, substruct1_nodir)
    sub2_dist = levenshtein_distance(protac_sub2, substruct2_nodir)

    # Return the substructure that is more similar to the PROTAC SMILES string
    if sub1_dist < sub2_dist:
        return substruct1, substruct2
    else:
        return substruct2, substruct1
    

def join_substructures(
        protac_smiles: str,
        e3_smiles: str,
        linker_smiles: str,
        poi_smiles: str,
) -> str:
    """Joins the substructures with the linker to create a PROTAC SMILES string.
    
    Args:
        protac_smiles: The PROTAC SMILES string.
        e3_smiles: The E3 ligand SMILES string.
        linker_smiles: The linker SMILES string.
        poi_smiles: The POI ligand SMILES string.
        
    Returns:
        The PROTAC SMILES string.
    """
    first_substruct, second_substruct = get_ordered_substruct(protac_smiles, e3_smiles, poi_smiles)
    return f'{first_substruct}.{linker_smiles}.{second_substruct}'


relevant_cols = [c for c in ds['standard']['train'].columns if 'smiles' in c.lower()]
for i, row in ds['standard']['train'][relevant_cols].sample(5, random_state=42).iterrows():
    protac_smiles = row.to_dict()['PROTAC SMILES']
    e3_smiles = row.to_dict()['E3 Binder SMILES with direction']
    linker_smiles = row.to_dict()['Linker SMILES with direction']
    poi_smiles = row.to_dict()['POI Ligand SMILES with direction']
    first_substruct, second_substruct = get_ordered_substruct(protac_smiles, e3_smiles, poi_smiles)

    new_protac, _ = reassemble_protac(
        randomize_smiles(poi_smiles),
        randomize_smiles(linker_smiles),
        randomize_smiles(e3_smiles),
        e3_bond_type='rand_uniform',
        poi_bond_type='rand_uniform',
    )


    # print('PROTAC:', protac_smiles)
    # print('PROTAC:', canonize_smiles(new_protac))
    # print('PROTAC:', new_protac)
    # print('SUBSTRUCT:', f'{first_substruct}.{linker_smiles}.{second_substruct}')
    # print('SUBSTRUCT:', randomize_smiles(f'{first_substruct}.{linker_smiles}.{second_substruct}'))
    # print('SUBSTRUCT:', randomize_smiles(f'{first_substruct}.{linker_smiles}.{second_substruct}'))
    # print('SUBSTRUCT:', randomize_smiles(f'{first_substruct}.{linker_smiles}.{second_substruct}'))
    # safe_display(Chem.MolFromSmiles(protac_smiles))
    # safe_display(Chem.MolFromSmiles(new_protac))
    # safe_display(Chem.MolFromSmiles(randomize_smiles(protac_smiles)))
    # safe_display(Chem.MolFromSmiles(randomize_smiles(protac_smiles)))
    # safe_display(Chem.MolFromSmiles(randomize_smiles(protac_smiles)))
    # safe_display(Chem.MolFromSmiles(first_substruct))
    # print('-' * 80)

## Augmentations

- 20x randomized SMILES per PROTACs


In [73]:
def get_unique_substructs(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    return {
        'e3': df['E3 Binder SMILES with direction'].unique(),
        'linker': df['Linker SMILES with direction'].unique(),
        'poi': df['POI Ligand SMILES with direction'].unique(),
    }


def get_substruct_prob(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    unique_substructs = get_unique_substructs(df)
    probs = {
        'e3': np.array([1 / df['E3 Binder SMILES with direction'].value_counts()[sub] for sub in unique_substructs['e3']]),
        'linker': np.array([1 / df['Linker SMILES with direction'].value_counts()[sub] for sub in unique_substructs['linker']]),
        'poi': np.array([1 / df['POI Ligand SMILES with direction'].value_counts()[sub] for sub in unique_substructs['poi']]),
    }
    # Normalize the probabilities so that they sum to 1
    return {k: v / v.sum() for k, v in probs.items()}


def get_fingerprint(smiles: str, morgan_fpgen = None, radius: int = 10, fpSize: int = 512) -> np.ndarray:
    """ Get the Morgan fingerprint of a molecule.
    
    Args:
        smiles (str): The SMILES string of the molecule.
        morgan_fpgen: The Morgan fingerprint generator.

    Returns:
        np.ndarray: The Morgan fingerprint.
    """
    if morgan_fpgen is None:
        morgan_fpgen = AllChem.GetMorganGenerator(
            radius=radius,
            fpSize=fpSize,
            includeChirality=True,
        )
    return morgan_fpgen.GetFingerprint(Chem.MolFromSmiles(smiles))


def get_recombined_df(
        df: Dict[str, pd.DataFrame],
        max_combinations: int = 5000,
        uniform_sampling: bool = True,
        similarity_threshold: float = 0.4,
        verbose: int = 0,
) -> pd.DataFrame:
    unique_substructs = get_unique_substructs(df['train'])

    # max_samples = int(max_combinations**(1/3))
    # if uniform_sampling:
    #     probs = {k: None for k in ['e3', 'linker', 'poi']}
    # else:
    #     probs = get_substruct_prob(df['train'])

    # combinations = product(
    #     np.random.choice(unique_substructs['e3'], min(max_samples, unique_substructs['e3'].size), replace=False, p=probs['e3']),
    #     np.random.choice(unique_substructs['linker'], min(max_samples, unique_substructs['e3'].size), replace=False, p=probs['linker']),
    #     np.random.choice(unique_substructs['poi'], min(max_samples, unique_substructs['e3'].size), replace=False, p=probs['poi']),
    # )
    # num_combinations = min(max_samples, unique_substructs['e3'].size) * min(max_samples, unique_substructs['e3'].size) * min(max_samples, unique_substructs['e3'].size)
    # if verbose:
    #     print(f'Maximum number of samples: {max_samples}')
    #     print(f'Number of actual combinations: {num_combinations:,}')

    np.random.shuffle(unique_substructs['e3']),
    np.random.shuffle(unique_substructs['linker']),
    np.random.shuffle(unique_substructs['poi']),
    combinations = product(
        unique_substructs['e3'],
        unique_substructs['linker'],
        unique_substructs['poi'],
    )

    # Precompute fingerprints of test PROTACs
    test_fps = df['test']['PROTAC SMILES'].apply(get_fingerprint).to_list()

    recombined_df = []
    for i, (e3, linker, poi) in tqdm(enumerate(combinations), total=max_combinations, desc='Recombining PROTACs'):
        if i >= max_combinations:
            break
        new_protac = None
        while not new_protac:
            try:
                new_protac, _ = reassemble_protac(
                    poi,
                    linker,
                    e3,
                    e3_bond_type='rand_uniform',
                    poi_bond_type='rand_uniform',
                )
            except:
                pass

        # Calculate bulk Tanimoto similarity
        fp = get_fingerprint(new_protac)
        similarities = DataStructs.BulkTanimotoSimilarity(fp, test_fps)
        avg_similarity = np.mean(similarities)

        if new_protac in df['test']['PROTAC SMILES'].values or avg_similarity > similarity_threshold:
            continue

        recombined_df.append({
            'PROTAC SMILES': new_protac,
            'E3 Binder SMILES with direction': e3,
            'Linker SMILES with direction': linker,
            'POI Ligand SMILES with direction': poi,
        })
        if i < 5 and verbose:
            print(f'{e3}.{linker}.{poi}')
            print(new_protac)
            print(randomize_smiles(new_protac))
            safe_display(Chem.MolFromSmiles(new_protac))
            print('-' * 80)

    return pd.DataFrame(recombined_df)

np.random.seed(42)

recombined_df = get_recombined_df(ds['standard'], max_combinations=1000)
safe_display(recombined_df)

recombined_df = get_recombined_df(ds['standard'], max_combinations=1000)
safe_display(recombined_df)

Recombining PROTACs:   0%|          | 0/1000 [00:00<?, ?it/s]

,PROTAC SMILES,E3 Binder SMILES with direction,Linker SMILES with direction,POI Ligand SMILES with direction
0,CC[C@@H](NC(=O)[C@@H]1C[C@H](OCC(=O)NCCCCNC(=O...,[*:2][C@H]1C[C@@H](C(=O)N[C@H](CC)c2ccccc2)N(C...,[*:2]OCC(=O)NCCCCNC(=O)[*:1],[*:1]N(C)Cc1ccc(-c2[nH]c3cc(F)cc4c3c2CCNC4=O)cc1
1,CC[C@@H](NC(=O)[C@@H]1C[C@H](OCC(=O)NCCCCNC(=O...,[*:2][C@H]1C[C@@H](C(=O)N[C@H](CC)c2ccccc2)N(C...,[*:2]OCC(=O)NCCCCNC(=O)[*:1],[*:1]O[C@H]1CC[C@H]2[C@@H]3CC[C@H]4CC(=O)CC[C@...
2,CC[C@@H](NC(=O)[C@@H]1C[C@H](OCC(=O)NCCCCNC(=O...,[*:2][C@H]1C[C@@H](C(=O)N[C@H](CC)c2ccccc2)N(C...,[*:2]OCC(=O)NCCCCNC(=O)[*:1],[*:1]CNC(=O)c1ccc(N2C(=S)N(c3ccc(C#N)c(C(F)(F)...
3,CC[C@@H](NC(=O)[C@@H]1C[C@H](OCC(=O)NCCCCNC(=O...,[*:2][C@H]1C[C@@H](C(=O)N[C@H](CC)c2ccccc2)N(C...,[*:2]OCC(=O)NCCCCNC(=O)[*:1],[*:1]c1ccc(Nc2ncc(F)c(NCCCN(C)C(=O)C3CCC3)n2)cc1
4,CC[C@@H](NC(=O)[C@@H]1C[C@H](OCC(=O)NCCCCNC(=O...,[*:2][C@H]1C[C@@H](C(=O)N[C@H](CC)c2ccccc2)N(C...,[*:2]OCC(=O)NCCCCNC(=O)[*:1],[*:1]OP(=O)(O)OC[C@@H]1O[C@H](n2c[n+](Cc3ccc(F...
...,...,...,...,...
824,CC[C@@H](NC(=O)[C@@H]1C[C@H](C(=O)CCCCCCCn2cc(...,[*:2][C@H]1C[C@@H](C(=O)N[C@H](CC)c2ccccc2)N(C...,[*:2]C(=O)CCCCCCCn1cc(CO[*:1])nn1,[*:1]CN1CCc2cc(Nc3ncc(C4CC4)c(NCCCNC(=O)C4CCC4...
825,CC[C@@H](NC(=O)[C@@H]1C[C@H](C(=O)CCCCCCCn2cc(...,[*:2][C@H]1C[C@@H](C(=O)N[C@H](CC)c2ccccc2)N(C...,[*:2]C(=O)CCCCCCCn1cc(CO[*:1])nn1,[*:1]c1ccc2nc(-c3ccc(NC)cc3)sc2c1
826,CC[C@@H](NC(=O)[C@@H]1C[C@H](C(=O)CCCCCCCn2cc(...,[*:2][C@H]1C[C@@H](C(=O)N[C@H](CC)c2ccccc2)N(C...,[*:2]C(=O)CCCCCCCn1cc(CO[*:1])nn1,[*:1]N1C(=O)N(C)c2ccc(-c3c(C)noc3C)cc2C1c1ccccc1
827,CC[C@@H](NC(=O)[C@@H]1C[C@H](C(=O)CCCCCCCn2cc(...,[*:2][C@H]1C[C@@H](C(=O)N[C@H](CC)c2ccccc2)N(C...,[*:2]C(=O)CCCCCCCn1cc(CO[*:1])nn1,[*:1]CN1C(=O)S/C(=C\c2ccc(Oc3ccc(C#N)cc3C(F)(F...


Recombining PROTACs:   0%|          | 0/1000 [00:00<?, ?it/s]

,PROTAC SMILES,E3 Binder SMILES with direction,Linker SMILES with direction,POI Ligand SMILES with direction
0,O=C(CN1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=...,[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O,[*:2]CC(=O)NCCCCCC[*:1],[*:1]COc1ccc(Nc2nccc(Nc3cnc4ccccc4c3)n2)cc1
1,CC(C)c1cnn2c(NCc3ccccc3)nc(OC3CCN(C=CCCCCCNC(=...,[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O,[*:2]CC(=O)NCCCCCC[*:1],[*:1]CN1CCC(Oc2nc(NCc3ccccc3)n3ncc(C(C)C)c3n2)CC1
2,C=Cc1cnc(Nc2ccc(CCCCCCNC(=O)CN3Cc4cc5c(cc4C3)C...,[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O,[*:2]CC(=O)NCCCCCC[*:1],[*:1]c1ccc(Nc2ncc(C=C)c(NCCCN(C)C(=O)C3CCC3)n2...
3,C[C@@]12C[C@H](NCCCCCCNC(=O)CN3Cc4cc5c(cc4C3)C...,[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O,[*:2]CC(=O)NCCCCCC[*:1],[*:1]N[C@H]1C[C@@]2(C)C[C@H](Oc3ccc(C#N)c(Cl)c...
4,O=C(CN1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=...,[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O,[*:2]CC(=O)NCCCCCC[*:1],[*:1]N1CCN(Cc2ccc(NC(=O)c3n[nH]cc3Nc3ncnc4[nH]...
...,...,...,...,...
814,CN1C[C@H](Nc2cnn(C)c(=O)c2Br)C[C@H](c2ccc(C(=O...,[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O,[*:2]CCOCCOCCC(=O)NCCN1CCN(CCCC(=O)[*:1])CC1,[*:1]c1ccc([C@H]2C[C@@H](Nc3cnn(C)c(=O)c3Br)CN...
815,O=C(CCCN1CCN(CCNC(=O)CCOCCOCCN2Cc3cc4c(cc3C2)C...,[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O,[*:2]CCOCCOCCC(=O)NCCN1CCN(CCCC(=O)[*:1])CC1,[*:1]CNc1ccc([N+](=O)[O-])c(Nc2ccc3[nH]ncc3c2)n1
816,COc1ccccc1S(=O)(=O)Nc1ccc2c3c(cc(C(=O)CCCN4CCN...,[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O,[*:2]CCOCCOCCC(=O)NCCN1CCN(CCCC(=O)[*:1])CC1,[*:1]c1cc2c3c(ccc(NS(=O)(=O)c4ccccc4OC)c3c1)N(...
817,CC/C(=C(\c1ccc(O)cc1)c1ccc(OCCN(C)C(=O)CCCN2CC...,[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O,[*:2]CCOCCOCCC(=O)NCCN1CCN(CCCC(=O)[*:1])CC1,[*:1]N(C)CCOc1ccc(/C(=C(/CC)c2ccccc2)c2ccc(O)c...


## Text Datasets

In [100]:
protac_col = 'PROTAC SMILES'
e3_col = 'E3 Binder SMILES with direction'
linker_col = 'Linker SMILES with direction'
poi_col = 'POI Ligand SMILES with direction'

max_rand_smiles_per_protac = 2

text_ds = {}
for config_name, datasets in ds.items():
    text_ds[config_name] = {}
    for split, dataset in datasets.items():
        text_df = ds[config_name][split].copy()

        # Add 'labels' column
        tqdm.pandas(desc=f'Joining {config_name} {split} substructures')
        text_df['labels'] = text_df.progress_apply(lambda x: join_substructures(x[protac_col], x[e3_col], x[linker_col], x[poi_col]), axis=1)

        # Rename 'PROTAC SMILES' column to 'text'
        text_df = text_df.rename(columns={'PROTAC SMILES': 'text'})

        text_ds[config_name][split] = text_df[['text', 'labels']]


    train_df = text_ds[config_name]['train']
    # TODO: Add configuration with randomized data
    randomized_df = []
    for _ in tqdm(range(max_rand_smiles_per_protac), desc=f'Randomizing {config_name} substructures'):
        tmp = train_df.copy()
        tmp['text'] = tmp['text'].apply(randomize_smiles)
        tmp['labels'] = tmp['labels'].apply(randomize_smiles)
        randomized_df.append(tmp)
    randomized_df = pd.concat(randomized_df).drop_duplicates()

    # TODO: Add configuration with recombined data
    recombined_df = get_recombined_df(datasets, max_combinations=len(randomized_df))
    tqdm.pandas(desc=f'Joining {config_name} recombined substructures', total=len(recombined_df))
    recombined_df['labels'] = recombined_df.progress_apply(lambda x: join_substructures(x[protac_col], x[e3_col], x[linker_col], x[poi_col]), axis=1)
    recombined_df = recombined_df.rename(columns={'PROTAC SMILES': 'text'})
    recombined_df = recombined_df.drop_duplicates()

    # TODO: Add configuration with recombined and randomized data
    rec_rand_df = recombined_df.copy()
    rec_rand_df['text'] = rec_rand_df['text'].apply(randomize_smiles)
    rec_rand_df['labels'] = rec_rand_df['labels'].apply(randomize_smiles)
    rec_rand_df = rec_rand_df.drop_duplicates()

    # NOTE: Make sure that all the above configurations have roughly the same number of samples, to better estimate the effect of data augmentations
    text_ds[f'{config_name}_randomized'] = {
        'train': pd.concat([train_df, randomized_df])[['text', 'labels']],
        'test': text_ds[config_name]['test'],
    }
    text_ds[f'{config_name}_recombined'] = {
        'train': pd.concat([train_df, recombined_df])[['text', 'labels']],
        'test': text_ds[config_name]['test'],
    }
    text_ds[f'{config_name}_rand_recombined'] = {
        'train': pd.concat([train_df, rec_rand_df])[['text', 'labels']],
        'test': text_ds[config_name]['test'],
    }

Joining standard train substructures:   0%|          | 0/2744 [00:00<?, ?it/s]

Joining standard test substructures:   0%|          | 0/313 [00:00<?, ?it/s]

Randomizing standard substructures:   0%|          | 0/2 [00:00<?, ?it/s]

Recombining PROTACs:   0%|          | 0/5488 [00:00<?, ?it/s]

Joining standard recombined substructures:   0%|          | 0/3040 [00:00<?, ?it/s]

...

Joining poi_unique train substructures:   0%|          | 0/2867 [00:00<?, ?it/s]

Joining poi_unique test substructures:   0%|          | 0/365 [00:00<?, ?it/s]

Randomizing poi_unique substructures:   0%|          | 0/2 [00:00<?, ?it/s]

Recombining PROTACs:   0%|          | 0/5734 [00:00<?, ?it/s]

Joining poi_unique recombined substructures:   0%|          | 0/5646 [00:00<?, ?it/s]

In [78]:
# Print the lengths of the datasets
for config_name, datasets in text_ds.items():
    print(f'{config_name}:')
    for split, dataset in datasets.items():
        print(f'  - {split} len:', len(dataset))

standard:
  - train len: 2744
  - test len: 313
standard_randomized:
  - train len: 9004
  - test len: 313
standard_recombined:
  - train len: 6145
  - test len: 313
standard_rand_recombined:
  - train len: 6145
  - test len: 313
hardest:
  - train len: 2867
  - test len: 286
hardest_randomized:
  - train len: 8587
  - test len: 286
hardest_recombined:
  - train len: 7758
  - test len: 286
hardest_rand_recombined:
  - train len: 7758
  - test len: 286
e3_unique:
  - train len: 2825
  - test len: 365
e3_unique_randomized:
  - train len: 10125
  - test len: 365
e3_unique_recombined:
  - train len: 9839
  - test len: 365
e3_unique_rand_recombined:
  - train len: 9839
  - test len: 365
linker_unique:
  - train len: 2874
  - test len: 366
linker_unique_randomized:
  - train len: 10194
  - test len: 366
linker_unique_recombined:
  - train len: 10043
  - test len: 366
linker_unique_rand_recombined:
  - train len: 10043
  - test len: 366
poi_unique:
  - train len: 2867
  - test len: 365
poi_un

## Hugging Face Datasets

If the data files are well organized in a directory, and a README configuration files is provided, then we can load the datasets directly from disk by _only knowing the directory path_:

In [99]:
from huggingface_hub import notebook_login

notebook_login()

In [106]:
from datasets import DatasetDict, Dataset

for config_name, datasets in text_ds.items():
    dataset_dict = DatasetDict({
        'train': Dataset.from_pandas(datasets['train'], preserve_index=False),
        'test': Dataset.from_pandas(datasets['test'], preserve_index=False),
    })
    dataset_dict.push_to_hub(
        'ailab-bio/PROTAC-Splitter-Dataset',
        config_name=config_name,
        private=True,
    )

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

In [96]:
from datasets import load_dataset, get_dataset_config_names, disable_caching


# Make text_datasets dir if not exists
dataset_dir = os.path.join(data_dir, 'text_datasets')
if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir, exist_ok=True)

config_template = '''- config_name: {config_name}
  data_files:
  - split: train
    path: "./{config_name}/train.csv"
  - split: test
    path: "./{config_name}/test.csv"
'''

# Create a README.md file
with open(os.path.join(dataset_dir, 'README.md'), 'w') as ds_config_file:
    # Write the header of the README.md file
    ds_config_file.write('---\n')
    ds_config_file.write('configs:\n')

    # TODO: The following needs to be replaced with the text-datasets
    for config_name, datasets in text_ds.items():
        # Write the current config template to the README.md file
        ds_config_file.write(config_template.format(config_name=config_name))

        # Create the config directory if it does not exist
        config_dir = os.path.join(dataset_dir, config_name)
        if not os.path.exists(config_dir):
            os.makedirs(config_dir, exist_ok=True)

        # Save the CSV datasets to the config directory
        for split, df in datasets.items():
            df.to_csv(os.path.join(config_dir, f'{split}.csv'), index=False)
            # safe_display(df.head())

    # Close the README.md file
    ds_config_file.write('---\n')

for config_name in get_dataset_config_names(dataset_dir):
    print(f'Loading {config_name} dataset at {dataset_dir}:')
    d = load_dataset(dataset_dir, name=config_name)

DatasetDict
Dataset.from_pandas(df)
    dataset.push_to_hub(
        "stevhliu/private_processed_demo",
        config_name=config_name,
        set_default=config_name == 'standard',
        private=True,
    )

    safe_display(d)
    safe_display(d['train']['text'][:5])
    safe_display(d['train']['labels'][:5])

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Loading standard dataset at /cephyr/users/ribes/Alvis/PROTAC-Splitter/data/text_datasets:


Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

DatasetDict({
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 365
    })
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 10154
    })
})

['C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NC(C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCC(=O)N[C@H](C(=O)N3C[C@H](O)C[C@H]3C(=O)N[C@@H](C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCOCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NC(C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCOCC(=O)N[C@H](C(=O)N3C[C@H](O)C[C@H]3C(=O)N[C@@H](C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1',
 'C=CC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCOCCOCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NC(C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1']

['[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1.[*:2]COCCOCCOCCN(C)C(=O)CCN[*:1].[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 '[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1.[*:2]COCCOCCOCCN(C)C(=O)CCN[*:1].[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 '[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1.[*:2]OCCOCCOCCOCCN(C)C(=O)CCN[*:1].[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 '[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1.[*:2]OCCOCCOCCOCCN(C)C(=O)CCN[*:1].[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C',
 '[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O)cc4ccccc34)c(F)c2n1.[*:2]OCCOCCOCCOCCOCCN(C)C(=O)CCN[*:1].[*:2]CC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c1ccc(-c2scnc2C)cc1)C(C)(C)C']

## Testing Tokenization

In [7]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_name = 'seyonec/PubChem10M_SMILES_BPE_450k'
model_name = 'entropy/roberta_zinc_480m'
# model_name = 'DeepChem/ChemBERTa-10M-MTR' # NOTE: Decoding is wrong, possibly due to missing symbols in the vocabulary

model = AutoModelForMaskedLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(2707, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-13): 14 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [8]:
print(model.config.max_length)
print(model.config.max_position_embeddings)
print(tokenizer.model_max_length)

tokenizer.model_max_length = model.config.max_position_embeddings
print(tokenizer.model_max_length)
print(tokenizer.max_model_input_sizes)

20
512
1000000000000000019884624838656
512
{'roberta-base': 512, 'roberta-large': 512, 'roberta-large-mnli': 512, 'distilroberta-base': 512, 'roberta-base-openai-detector': 512, 'roberta-large-openai-detector': 512}


In [9]:
# Tokenize the top-10 longest PROTAC SMILES strings
df = recombined_df
df = ds['standard']['train']
df = ds['standard']['test']

idx = df['PROTAC SMILES'].str.len().nlargest(10).index
protac_smiles = list(df.loc[idx, 'PROTAC SMILES'])

# Print the length of each PROTAC SMILES string
for i, smiles in enumerate(protac_smiles):
    print(f'{i + 1}: {len(smiles)}')

# Tokenize the PROTAC SMILES strings
tokenized = tokenizer(protac_smiles, return_tensors='pt', padding=True, truncation=True)

# Print the length of the tokenized PROTAC SMILES strings
for i, input_ids in enumerate(tokenized['input_ids']):
    print(f'{i + 1}: {len(input_ids)}')

# Decode the tokenized PROTAC SMILES strings and check if they match the original strings
decoded = tokenizer.batch_decode(tokenized['input_ids'], skip_special_tokens=True)
for i, (smiles, dec) in enumerate(zip(protac_smiles, decoded)):
    print(f'{i + 1}: {smiles == dec}')
    if not smiles == dec:
        print(f'Original: {smiles}')
        print(f'Decoded:  {dec}')

1: 236
2: 227
3: 219
4: 213
5: 212
6: 212
7: 209
8: 205
9: 200
10: 190
1: 167
2: 167
3: 167
4: 167
5: 167
6: 167
7: 167
8: 167
9: 167
10: 167
1: True
2: True
3: True
4: True
5: True
6: True
7: True
8: True
9: True
10: True


In [13]:
from transformers import AutoTokenizer, EncoderDecoderModel
from typing import Optional

def get_model(
    pretrained_encoder: str = "seyonec/ChemBERTa-zinc-base-v1",
    pretrained_decoder: str = "seyonec/ChemBERTa-zinc-base-v1",
    max_length: Optional[int] = 512,
    tie_encoder_decoder: bool = False,
):
    
    commit_hash = '0ba58478f467056fe33003d7d91644ecede695a7'
    bert2bert = EncoderDecoderModel.from_encoder_decoder_pretrained(
        pretrained_encoder,
        pretrained_decoder,
        tie_encoder_decoder=tie_encoder_decoder,
        trust_remote_code=True,
        revision=commit_hash,
    )
    print(f"Number of parameters: {bert2bert.num_parameters():,}")
    tokenizer = AutoTokenizer.from_pretrained(pretrained_encoder)
    # Tokenizer configs
    bert2bert.config.decoder_start_token_id = tokenizer.cls_token_id
    bert2bert.config.eos_token_id = tokenizer.sep_token_id
    bert2bert.config.pad_token_id = tokenizer.pad_token_id
    bert2bert.config.vocab_size = bert2bert.config.encoder.vocab_size
    # Generation configs
    # NOTE: See full list of configurations can be found here: https://huggingface.co/docs/transformers/v4.33.3/en/main_classes/text_generation#transformers.GenerationConfig
    bert2bert.encoder.config.max_length = max_length
    bert2bert.decoder.config.max_length = max_length
    # bert2bert.config.min_length = 20

    # # NOTE: Never sample, i.e., always return the token w/ highest probability
    bert2bert.config.do_sample = False
    # bert2bert.config.do_sample = True
    # bert2bert.config.num_beams = 5
    # bert2bert.config.top_k = 20
    
    # bert2bert.config.max_new_tokens = 514
    # bert2bert.config.early_stopping = True
    # bert2bert.config.length_penalty = 2.0
    # # bert2bert.config.no_repeat_ngram_size = 3 # Default: 0
    
    return bert2bert


model_name = 'seyonec/PubChem10M_SMILES_BPE_450k'

bert2bert = get_model(
    pretrained_encoder=model_name,
    pretrained_decoder=model_name,
)
bert2bert

Some weights of the model checkpoint at seyonec/PubChem10M_SMILES_BPE_450k were not used when initializing RobertaModel: ['lm_head.decoder.bias', 'lm_head.decoder.weight', 'lm_head.layer_norm.bias', 'lm_head.dense.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForCausalLM were not initialized from the model checkpoint at seyonec/PubChem10M_SMILES_BPE_450k and are newly initialized: ['roberta.encoder.layer.1.crossattention.output.dense.bias', 'roberta.encoder.layer.2.crossattention.out

Number of parameters: 181,135,648


EncoderDecoderModel(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L